# Paper 3 — E1: Environment-Dependent (Gated) Adaptation

**Experiment E1** (problem statement v0.2, H2-H3): three update policies for the UE-side filter —
**never** (freeze after warm-up), **always**, and **gated** (update only when the per-UE innovation
energy exceeds a threshold) — traced as an **error vs update-fraction curve** per regime.
The expectation from E0: in the static regime *never* loses little (a fixed filter almost suffices),
in the dynamic regime it collapses, and *gated* recovers most of the *always* gain at a fraction of
the updates. This curve is the bridge from H2 to the E2 energy Pareto.

Input: the M2 HDF5 datasets (static Ns0, dynamic Ns10, and dynamic Ns30 when generated).
Filter: NLMS, p = 4, mu = 0.5 (the deployable low-cost candidate from E0); set ALGO = 'rls' for the
RLS variant. CPU-only, minutes.

In [ ]:
# %% [1] Setup
import os, json, uuid, glob, datetime
import numpy as np, h5py
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/paper3'
M2_DIR = os.path.join(BASE, 'm2')
E1_DIR = os.path.join(BASE, 'e1'); os.makedirs(E1_DIR, exist_ok=True)

def drive_canary(path=E1_DIR):
    token = str(uuid.uuid4()); c = os.path.join(path, '.canary')
    with open(c, 'w') as f: f.write(token)
    with open(c) as f: assert f.read() == token, 'DRIVE CANARY FAILED: phantom mount.'
    os.remove(c); print('Drive canary OK:', path)

drive_canary()

In [ ]:
# %% [2] Configuration
ALGO = 'nlms'                   # 'nlms' (deployable low-cost) or 'rls' (upper reference)
P_ORDER = 4
MU_NLMS = 0.5                   # E0 sweep: best tracking step in the dynamic regime
RLS_LAMBDA = 0.995
RLS_DELTA = 1e-2
RCOND = 1e-4                    # Wiener pinv truncation, calibrated in E0
EMA_ALPHA = 0.05                # smoothing of the per-UE innovation ratio (~20-sample memory)
THETAS = (0.002, 0.005, 0.01, 0.02, 0.05, 0.1)   # gate thresholds on the smoothed innovation ratio
OUTAGE_MAX = 0.10               # drop a UE if its stored serving link is in outage more than this
WARMUP_FRAC = 0.1               # always adapt during warm-up (convergence), policy applies after
TRAIN_FRAC = 0.5                # evaluate on the second half, as in E0
SEED = 2026
np.random.seed(SEED)

In [ ]:
# %% [3] Load M2 datasets (Doppler format only; Ns30 joins automatically once generated)
def load_series(h5path):
    with h5py.File(h5path, 'r') as f:
        if 'fine_dt' not in f.attrs:
            return None
        H = f['csi'][:]
        meta = dict(regime=str(f.attrs['regime']), n_scat=int(f.attrs['n_scatterers']),
                    dt=float(f.attrs['fine_dt']),
                    speed=str(f.attrs.get('speed_label', 'walk')))
    if H.ndim == 5:                                   # shakedown layout [T, U, rx_ant, A, S]
        E = (np.abs(H)**2).sum(axis=(2, 4)); serve = E.mean(0).argmax(1)
        H = np.stack([H[:, u, :, serve[u], :] for u in range(H.shape[1])], 1)
    # Drop UEs whose stored serving link goes into outage (>20 dB below the best array) for more
    # than OUTAGE_MAX of the run: those UEs walk out of their array's coverage and would need a
    # handover, which is outside the scope of E0/E1 (single-link filtering). Typically 1 UE in 20.
    with h5py.File(h5path, 'r') as f:
        if 'energy_dB' in f:
            Edb = f['energy_dB'][:]; serve_i = f['serving_array'][:].astype(np.int64)
            deficit = Edb.max(2) - np.take_along_axis(Edb, serve_i[None, :, None], axis=2)[:, :, 0]
            out_frac = (deficit > 20).mean(0)
            keep = out_frac <= OUTAGE_MAX
            if (~keep).any():
                dropped = [(int(u), round(float(out_frac[u])*100)) for u in np.where(~keep)[0]]
                print(f'  dropped UEs (outage% ): {dropped}')
            H = H[:, keep]
    U = H.shape[1]                                    # now [T, U, rx_ant, S] in both layouts
    X = H.reshape(H.shape[0], U, -1)
    X = X / (np.sqrt((np.abs(X)**2).mean(axis=(0, 2), keepdims=True)) + 1e-30)
    return X.astype(np.complex64), meta

DATA = {}
for p_ in sorted(glob.glob(os.path.join(M2_DIR, 'paper3_M2*.h5'))):
    out = load_series(p_)
    if out is None:
        print('SKIPPED (old format):', os.path.basename(p_)); continue
    X, meta = out
    key = os.path.basename(p_).replace('paper3_M2_', '').replace('.h5', '')
    key = key.replace(f"_U{X.shape[1]}", '').replace('_seed2026', '')
    assert key not in DATA, f'duplicate key {key}: two runs map to the same name'
    DATA[key] = (X, meta)
    print(f"{key:14s} | X {X.shape} [T, U, C]")
assert DATA, 'no usable M2 datasets found'

In [ ]:
# %% [4] Gated adaptive filter
# One filter per (UE, component); ONE gate per UE: the running (EMA) innovation ratio of the whole
# link decides whether that UE spends update energy this sample. theta=None -> always;
# theta=np.inf -> never (freeze after warm-up).
def run_policy(X, p, theta, algo=ALGO):
    T, U, C = X.shape
    Xf = X.reshape(T, U*C)
    w = np.zeros((p, U*C), np.complex128)
    if algo == 'rls':
        P = np.tile((1.0/RLS_DELTA)*np.eye(p, dtype=np.complex128), (U*C, 1, 1))
    t_warm = int(WARMUP_FRAC*T)
    ema = np.full(U, 1.0)
    e_all, d_all, upd = [], [], np.zeros(T, np.float32)
    for t in range(p, T):
        u = Xf[t-p:t][::-1].astype(np.complex128)
        e = Xf[t] - (np.conj(w)*u).sum(0)
        e_all.append(e); d_all.append(Xf[t])
        inst = (np.abs(e.reshape(U, C))**2).sum(1)/((np.abs(Xf[t].reshape(U, C))**2).sum(1) + 1e-30)
        ema = (1-EMA_ALPHA)*ema + EMA_ALPHA*inst
        if theta is None or t < t_warm:
            gate = np.ones(U, bool)
        elif np.isinf(theta):
            gate = np.zeros(U, bool)
        else:
            gate = ema > theta
        upd[t] = gate.mean()
        if gate.any():
            g = np.repeat(gate, C)
            if algo == 'nlms':
                w[:, g] += MU_NLMS * u[:, g] * np.conj(e[g]) / ((np.abs(u[:, g])**2).sum(0) + 1e-6)
            else:
                ug, Pg = u[:, g], P[g]
                Pu = np.einsum('kij,jk->ik', Pg, ug)
                den = RLS_LAMBDA + (np.conj(ug)*Pu).sum(0)
                gain = Pu/den
                w[:, g] += gain*np.conj(e[g])
                uHP = np.einsum('ik,kij->jk', np.conj(ug), Pg)
                P[g] = (Pg - np.einsum('ik,jk->kij', gain, uHP))/RLS_LAMBDA
    e_all, d_all = np.array(e_all), np.array(d_all)
    t_split = int(TRAIN_FRAC*T)
    mask = np.zeros(len(e_all), bool); mask[t_split-p:] = True
    nm = float((np.abs(e_all[mask])**2).sum()/((np.abs(d_all[mask])**2).sum() + 1e-30))
    frac = float(upd[t_split:].mean())            # update fraction on the evaluation half
    return nm, frac

In [ ]:
# %% [4b] Fair 'never' baseline: the BEST fixed filter, not a frozen adaptive one
# E1 v1 implemented 'never' as NLMS frozen after warm-up, which reached only 0.0876 NMSE in the
# static set while E0's batch Wiener reached 0.0018 on the same data: that overstates the need for
# adaptation. The honest no-adaptation baseline is the batch Wiener fitted on the training half.
def run_fixed_wiener(X, p):
    T, U, C = X.shape
    Xf = X.reshape(T, U*C).astype(np.complex128)
    t_split = int(TRAIN_FRAC*T)
    K = Xf.shape[1]
    Xtr = Xf[:t_split]
    r = np.zeros((p+1, K), np.complex128)
    r[0] = (np.abs(Xtr)**2).mean(0)
    for k in range(1, p+1):
        r[k] = (Xtr[k:]*np.conj(Xtr[:-k])).mean(0)
    R = np.zeros((K, p, p), np.complex128)
    for i in range(p):
        for j in range(p):
            R[:, i, j] = np.conj(r[i-j]) if i >= j else r[j-i]
    q = np.conj(r[1:p+1]).T[..., None]
    w = (np.linalg.pinv(R, rcond=RCOND) @ q)[..., 0].T
    e, d = [], []
    for t in range(p, T):
        u = Xf[t-p:t][::-1]
        e.append(Xf[t] - (np.conj(w)*u).sum(0)); d.append(Xf[t])
    e, d = np.array(e), np.array(d)
    mask = np.zeros(len(e), bool); mask[t_split-p:] = True
    return float((np.abs(e[mask])**2).sum()/((np.abs(d[mask])**2).sum() + 1e-30)), 0.0

In [ ]:
# %% [5] Policy sweep per dataset
import time
rows = []
for key, (X, meta) in sorted(DATA.items()):
    for name, theta in ([('always', None), ('never_frozen', np.inf), ('never_wiener', 'wiener')]
                        + [(f'gated@{t}', t) for t in THETAS]):
        t0 = time.time()
        if theta == 'wiener':
            nm, frac = run_fixed_wiener(X, P_ORDER)
        else:
            nm, frac = run_policy(X, P_ORDER, theta)
        rows.append(dict(dataset=key, policy=name,
                         theta=None if theta in (None, 'wiener') else float(theta),
                         nmse=nm, update_frac=frac))
        print(f'{key:14s} {name:12s} | NMSE {nm:.4f} | updates {frac*100:5.1f}% | {time.time()-t0:4.1f} s')

In [ ]:
# %% [6] Error vs update-fraction curve + H2/H3 checks
import pandas as pd
df = pd.DataFrame(rows)
plt.figure(figsize=(7, 5))
for key, g in df.groupby('dataset'):
    gg = g.sort_values('update_frac')
    plt.plot(gg.update_frac*100, gg.nmse, 'o-', label=key)
    for _, r in gg.iterrows():
        if r.policy in ('always', 'never_frozen', 'never_wiener'):
            plt.annotate(r.policy, (r.update_frac*100, r.nmse), fontsize=7,
                         textcoords='offset points', xytext=(4, 4))
plt.xlabel('update fraction on evaluation half [%]'); plt.ylabel('NMSE'); plt.yscale('log')
plt.grid(alpha=0.3); plt.legend(fontsize=8); plt.title(f'E1: error vs updates ({ALGO}, p={P_ORDER})')
plt.tight_layout(); plt.savefig(os.path.join(E1_DIR, f'e1_pareto_{ALGO}.png'), dpi=150); plt.show()

print()
for key, g in df.groupby('dataset'):
    alw = g[g.policy == 'always'].iloc[0]
    nev = g[g.policy == 'never_wiener'].iloc[0]          # the FAIR no-adaptation baseline
    frz = g[g.policy == 'never_frozen'].iloc[0]

    ok = None
    for _, r in g[g.policy.str.startswith('gated')].sort_values('update_frac').iterrows():
        if r.nmse <= 2*alw.nmse and r.update_frac <= 0.5:     # within 2x of always-adapt
            ok = r; break
    # H2 is about how much error a UE is LEFT WITH at a given update budget, not about the ratio:
    # in the quasi-static regime the ratio is huge (98x) over negligible absolute errors.
    cheap = g[(g.policy.str.startswith('gated')) & (g.update_frac <= 0.05)].nmse.min()
    print(f'{key:24s} | no adaptation: Wiener {nev.nmse:.4f} / frozen {frz.nmse:.4f} | '
          f'<=5% updates {cheap:.4f} | always {alw.nmse:.4f}')
    print(f'{"":24s} | [H2] no-adaptation error already low (best fixed < 0.01): '
          f'{"YES - static-like, adaptation optional" if nev.nmse < 0.01 else "NO - dynamic-like, adaptation needed"}')
    if ok is not None:
        print(f'{"":24s} | [H3] gated within 2x of always at <=50% updates: PASS '
              f'({ok.policy}: {ok.nmse:.4f} at {ok.update_frac*100:.1f}% updates)')
    elif nev.nmse < 0.01:
        print(f'{"":24s} | [H3] N/A - adaptation is unnecessary here, so the 2x-of-always target '
              f'({2*alw.nmse:.5f}) is below what any policy needs to reach')
    else:
        print(f'{"":24s} | [H3] gated within 2x of always at <=50% updates: FAIL')

In [ ]:
# %% [7] Save + backup
drive_canary()
df.to_csv(os.path.join(E1_DIR, f'e1_results_{ALGO}.csv'), index=False)
with open(os.path.join(E1_DIR, f'e1_results_{ALGO}_{datetime.datetime.now():%Y%m%d_%H%M}.json'), 'w') as f:
    json.dump(dict(rows=rows, config=dict(algo=ALGO, p=P_ORDER, mu=MU_NLMS, rls_lambda=RLS_LAMBDA,
                                          ema_alpha=EMA_ALPHA, thetas=list(THETAS),
                                          warmup_frac=WARMUP_FRAC, seed=SEED)), f, indent=2)
import shutil
zip_path = shutil.make_archive('/content/paper3_e1_backup', 'zip', E1_DIR)
from google.colab import files as colab_files
colab_files.download(zip_path)
print('Saved.')